<a href="https://colab.research.google.com/github/Ratludu/Backpack-Prediction-Challenge/blob/main/Backpack_Prices_LB_38.84972.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [2]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggleusername')
os.environ['KAGGLE_KEY'] = userdata.get('kaggleapi')

competition = 'playground-series-s5e2'

!kaggle competitions download -c {competition}

!unzip "{competition}.zip"

 98% 91.0M/92.7M [00:04<00:00, 24.5MB/s]
100% 92.7M/92.7M [00:05<00:00, 19.2MB/s]
Archive:  playground-series-s5e2.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
  inflating: training_extra.csv      


In [ ]:
!pip install dask-cuda==24.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.4/134.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.5/244.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2024.10.0
    Uninstalling dask-2024.10.0:
      Successfully uninstalled dask-2024.10.0


In [3]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 577, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 577 (delta 116), reused 82 (delta 82), pack-reused 434 (from 3)
Receiving objects: 100% (577/577), 188.95 KiB | 23.62 MiB/s, done.
Resolving deltas: 100% (290/290), done.
Using Python 3.11.11 environment at: /usr
Resolved 2 packages in 86ms
Prepared 2 packages in 8ms
Installed 2 packages in 3ms
 + nvidia-ml-py==12.570.86
 + pynvml==12.0.0
Installing RAPIDS remaining 24.12.* libraries
Using Python 3.11.11 environment at: /usr
Resolved 154 packages in 9.61s
 Downloaded ucx-py-cu12
 Downloaded cuspatial-cu12
 Downloaded libcuspatial-cu12
 Downloaded libucx-cu12
 Downloaded datashader
 Downloaded dask
 Downloaded cucim-cu12
 Downloaded scikit-image
 Downloaded raft-dask-cu12
 Downloaded cuml-cu12
 Downloaded cugraph-cu12
 Downloaded cuvs-cu12
Prepared 30 packages in 26.96s
Uninstalled 3 packages in 43ms
I

In [5]:
!pip install catboost
!pip install optuna
!pip install scikit-learn
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.5 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
from numpy import random
from cuml.preprocessing import TargetEncoder
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [7]:
class config:
    # data links
    train_link = "train.csv"
    train_ex_link = "training_extra.csv"
    test_link = "test.csv"
    sub_link = "sample_submission.csv"

    # create folds config

    n_splits = 25

    # Ignore Columns

    col_ignore = ["id", "Price"]
    num_cols = ["Weight Capacity (kg)"]
    # target

    submit = True

    target = "Price"

    add_original = False

In [8]:
def rmse(y_true, y_pred):
    error = 0

    for yt, yp in zip(y_true, y_pred):
        error += (yt - yp) ** 2

    m = np.sqrt(error / len(y_true))

    return m

In [9]:
def random_columns(columns):

  # Generate random number for how many columns we want to concat
  rand_num = np.random.randint(2,5)

  # choose the columns from the list of columns with no repeats
  rand_cols = []
  for i in range(rand_num):
    col = np.random.choice(columns)
    while col in rand_cols:
      col = np.random.choice(columns)
    rand_cols.append(col)

  # return a list of the columns

  return "-".join(col for col in rand_cols),rand_cols


In [10]:
train = pd.read_csv(config.train_link)
train_ex = pd.read_csv(config.train_ex_link)
test = pd.read_csv(config.test_link)

In [11]:
train = pd.concat([train,train_ex], axis = 0, ignore_index = True)

In [12]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
oof = np.zeros(len(train))
preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    x_train["weight_log"] = np.log1p(x_train["Weight Capacity (kg)"])
    x_val["weight_log"] = np.log1p(x_val["Weight Capacity (kg)"])
    x_test["weight_log"] = np.log1p(x_test["Weight Capacity (kg)"])

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    # median

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'median')

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_median'] = TE.transform(x_train[col])
      x_val[f'{col}_median'] = TE.transform(x_val[col])
      x_test[f'{col}_median'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_median'] = TE.transform(x_train[col])
        x_val[f'{col}_median'] = TE.transform(x_val[col])
        x_test[f'{col}_median'] = TE.transform(x_test[col])
    # var

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'var')

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_var'] = TE.transform(x_train[col])
      x_val[f'{col}_var'] = TE.transform(x_val[col])
      x_test[f'{col}_var'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_var'] = TE.transform(x_train[col])
        x_val[f'{col}_var'] = TE.transform(x_val[col])
        x_test[f'{col}_var'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('str')
        x_val[cat] = x_val[cat].astype('str')
        x_test[cat] = x_test[cat].astype('str')

    print(x_train.columns)

    model = CatBoostRegressor(
                              learning_rate = 0.11509572776170199,
                              l2_leaf_reg=5,
                              task_type = "GPU",
                              grow_policy = 'Lossguide',
                              random_state = 42,
                              cat_features = cats,
                              verbose = 250,
                              loss_function='RMSE')

    model.fit(x_train, y_train)

    val_preds = model.predict(x_val)

    oof[test_idx] = val_preds

    preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')

print(f"The average CV is {np.average(m)}")

Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'weight_log',
       'size-laptop compartment_fe_TE', 'Color-waterproof_fe_TE',
       'weightcapacity-color_fe_TE', 'Brand_TE', 'Material_TE', 'Size_TE',
       'Compartments_TE', 'Laptop Compartment_TE', 'Waterproof_TE', 'Style_TE',
       'Color_TE', 'Weight Capacity (kg)_TE',
       'size-laptop compartment_fe_median', 'Color-waterproof_fe_median',
       'weightcapacity-color_fe_median', 'Brand_median', 'Material_median',
       'Size_median', 'Compartments_median', 'Laptop Compartment_median',
       'Waterproof_median', 'Style_median', 'Color_median',
       'Weight Capacity (kg)_median', 'size-laptop compartment_fe_var',
       'Color-waterproof_fe_var', 'weightcapacity-color_fe_var', 'Brand_var',
       'Material_var', 'Size_var', 'Compartments_var',

In [ ]:
from google.colab import runtime
runtime.unassign()

In [13]:
submission = pd.read_csv(config.sub_link)
submission[config.target] = preds
submission.to_csv("submission.csv", index = False)

submission

,id,Price
0,300000,81.217270
1,300001,82.863034
2,300002,87.477715
3,300003,77.459452
4,300004,78.898365
...,...,...
199995,499995,81.671461
199996,499996,73.658814
199997,499997,82.545395
199998,499998,82.072544


In [14]:
if config.submit:
  !kaggle competitions submit -c {competition} -f submission.csv -m 'Submission with lasso'

100% 4.74M/4.74M [00:02<00:00, 1.86MB/s]
Successfully submitted to Backpack Prediction Challenge

In [ ]:
!kaggle competitions submissions -c {competition}